# Tile index: check the remote agent's metadata reader

`read_tile_index(DATA_DIR)` (written by the remote agent, task T001) reads the metadata CSV and
pairs each SEM image with its `.oas` in `Contour/`. It returns one record per tile.

This notebook shows what it returns, then looks at one tile:
1. the tile table;
2. one SEM image, to see whether it has a **data bar** (a strip with text/scale bar at the bottom);
3. the design and SEM contacts of that tile in **mask coordinates** (tile-local + tile centre).

Set `DATA_DIR` below and run all cells.

In [ ]:
DATA_DIR = r"path\to\data"   # folder with the .jpg files, the metadata .csv and Contour/
TILE = 0                      # which tile to look at (index in the table)
LAYER = (1, 0)                # (layer, datatype) of the contacts in the .oas
DATABAR_ROWS = 0              # set after looking at the image in step 2

In [ ]:
import dataclasses

import matplotlib.pyplot as plt
import pandas as pd

from affine_ransac.features.contact import detect_contacts
from affine_ransac.geometry.frames import pixel_to_tile_nm
from affine_ransac.io.design import load_layout, read_contacts
from affine_ransac.io.metadata import read_tile_index
from affine_ransac.io.sem_image import crop_databar, load_sem_image

## 1. Tile table

In [ ]:
tiles = read_tile_index(DATA_DIR)

table = pd.DataFrame([dataclasses.asdict(t) for t in tiles]).drop(columns="extra")
table["image_path"] = table.image_path.map(lambda p: p.name)
table["oas_path"] = table.oas_path.map(lambda p: p.name if p else None)

print(f"{len(tiles)} tiles, {table.oas_path.isna().sum()} without an .oas file")
table

## 2. One SEM image: is there a data bar?

Look at the bottom of the image. If there is a strip with text or a scale bar, note the
row (y, in px) where it starts, and set `DATABAR_ROWS` = image height − that row.

In [ ]:
tile = tiles[TILE]
raw = load_sem_image(tile.image_path)
print(f"Tile {tile.tile_id}: image file is {raw.shape[1]} x {raw.shape[0]} px; "
      f"metadata says {tile.image_width_px} x {tile.image_height_px} px")

plt.figure(figsize=(8, 8))
plt.imshow(raw, cmap="gray")
plt.title(f"{tile.image_path.name}  (x right, y down, px)")
plt.show()

## 3. This tile's contacts in mask coordinates

Design: `.oas` positions (centred on (0, 0)) + tile centre.
SEM: detected centres → tile-local nm → + tile centre.
The two sets of points should lie on top of each other.

In [ ]:
design_local, _ = read_contacts(load_layout(tile.oas_path), *LAYER)

image = crop_databar(raw, DATABAR_ROWS)
found = detect_contacts(image)
sem_local = pixel_to_tile_nm(found.centers, image.shape, tile.fov_x_nm / image.shape[1])

center = (tile.center_x_nm, tile.center_y_nm)
design_mask = design_local + center
sem_mask = sem_local + center

print(f"Tile centre (mask): x = {center[0]:.1f} nm, y = {center[1]:.1f} nm")
print(f"Design contacts: {len(design_mask)}, SEM contacts: {len(sem_mask)}")

fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(design_mask[:, 0], design_mask[:, 1], marker="s", facecolors="none", edgecolors="tab:blue", label="design (.oas)")
ax.scatter(sem_mask[:, 0], sem_mask[:, 1], marker="+", color="tab:red", label="SEM")
ax.set_aspect("equal")
ax.set_xlabel("mask x (nm)")
ax.set_ylabel("mask y (nm)")
ax.legend()
plt.show()